# Оценка алгоритмов веломаршрутизации по STADTRADELN

Ноутбук использует **наблюдённую популярность дорожных сегментов** STADTRADELN 2024 для Констанца: число GPS-поездок, сопоставленных каждому сегменту. Он скачивает GeoPackage, показывает потоки на карте и оценивает маршруты, построенные вашим алгоритмом.

Источник: [Offene Daten Konstanz — STADTRADELN](https://offenedaten-konstanz.de/dataset/stadtradeln), лицензия CC BY-NC-SA 4.0.

> Это данные участников трёхнедельной кампании, а не полный городской велопоток и не среднесуточная интенсивность. Популярность нужно рассматривать вместе с длиной, временем, уклоном и безопасностью.

## 1. Установка и настройки

Вход вашего алгоритма — GeoJSON или GeoPackage с маршрутами `LineString`. Обязательные поля: `od_id` и `algorithm`. Для каждой OD-пары должна быть одна строка на алгоритм, включая baseline с именем `shortest`.

In [1]:
%pip install -q geopandas pyogrio shapely folium mapclassify requests tqdm pandas numpy plotly

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

DATA_DIR = Path("data/stadtradeln_konstanz")
DATA_DIR.mkdir(parents=True, exist_ok=True)
VOLUME_URL = ("https://offenedaten-konstanz.de/sites/default/files/"
              "Verkehrsmengen%202.0_SR%202024_Konstanz_UTM32_je_Wochentag_gesamt.zip")
ARCHIVE_PATH = DATA_DIR / "stadtradeln_konstanz_2024_traffic_volumes.zip"
ROUTES_PATH = Path("candidate_routes.geojson")  # результат вашего алгоритма
ROUTES_LAYER = None       # укажите слой, если ROUTES_PATH — GeoPackage
SAMPLE_STEP_M = 10.0      # шаг точек вдоль маршрута
MAX_MATCH_DISTANCE_M = 20.0
BASELINE_ALGORITHM = "shortest"

## 2. Скачать и прочитать observed edge popularity

In [3]:
import requests
from tqdm.auto import tqdm
from zipfile import ZipFile

def download_file(url: str, destination: Path) -> Path:
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Уже скачано: {destination}")
        return destination
    temporary = destination.with_suffix(destination.suffix + ".part")
    with requests.get(url, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0)) or None
        with temporary.open("wb") as output, tqdm(total=total, unit="B", unit_scale=True) as bar:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    output.write(chunk); bar.update(len(chunk))
    temporary.replace(destination)
    return destination

download_file(VOLUME_URL, ARCHIVE_PATH)
with ZipFile(ARCHIVE_PATH) as archive:
    archive.extractall(DATA_DIR)
gpkg_paths = sorted(DATA_DIR.rglob("trafficvolumes.gpkg"))
print(*gpkg_paths, sep="\n")

d:\PythonProjects\CicleGpx\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 3.66M/3.66M [00:03<00:00, 1.12MB/s]

data\stadtradeln_konstanz\Verkehrsmengen 2.0_SR 2024_Konstanz_UTM32_gesamt\trafficvolumes.gpkg
data\stadtradeln_konstanz\Verkehrsmengen 2.0_SR 2024_Konstanz_UTM32_je_Wochentag\trafficvolumes.gpkg


In [4]:
import geopandas as gpd
import pandas as pd
import numpy as np

total_path = next(p for p in gpkg_paths if p.parent.name.endswith("_gesamt"))
weekday_path = next(p for p in gpkg_paths if p.parent.name.endswith("_je_Wochentag"))
observed = gpd.read_file(total_path, layer="TrafficEdge")
weekday = gpd.read_file(weekday_path, layer="TrafficEdge")
assert observed.crs.to_epsg() == 25832  # метры
display(observed.head())
display(observed["number_of_matched_trips"].describe(percentiles=[.5, .75, .9, .95, .99]))

d:\PythonProjects\CicleGpx\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: GPKG: unrecognized user_version=0x00000000 (0) on 'data\stadtradeln_konstanz\Verkehrsmengen 2.0_SR 2024_Konstanz_UTM32_gesamt\trafficvolumes.gpkg'
  return ogr_read(
d:\PythonProjects\CicleGpx\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: GPKG: unrecognized user_version=0x00000000 (0) on 'data\stadtradeln_konstanz\Verkehrsmengen 2.0_SR 2024_Konstanz_UTM32_je_Wochentag\trafficvolumes.gpkg'
  return ogr_read(


,osm_way_id,number_of_matched_trips,geometry
0,479949911,25,"LINESTRING (508293.172 5287385.236, 508309.151..."
1,24034559,6,"LINESTRING (506751.517 5287361.799, 506763.796..."
2,147361632,17,"LINESTRING (507167.447 5288742.858, 507198.08 ..."
3,5215356,23,"LINESTRING (507470.749 5287025.69, 507484.985 ..."
4,1198789844,21,"LINESTRING (507248.565 5289033.771, 507401.831..."


count    42291.000000
mean        75.763094
std        198.407051
min          0.000000
50%         13.000000
75%         61.000000
90%        184.000000
95%        334.000000
99%       1017.000000
max       2628.000000
Name: number_of_matched_trips, dtype: float64

In [ ]:
observed.explore(
    column="number_of_matched_trips", scheme="FisherJenks", k=7, cmap="viridis",
    tiles="CartoDB positron", tooltip=["osm_way_id", "number_of_matched_trips"],
    style_kwds={"weight": 3}, legend_kwds={"caption": "Matched trips"},
)

d:\PythonProjects\CicleGpx\.venv\Lib\site-packages\geopandas\explore.py:388: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(scale_factor="{r}")
d:\PythonProjects\CicleGpx\.venv\Lib\site-packages\geopandas\explore.py:493: UserWarning: Numba not installed. Using slow pure python version.
  binning = classify(


## 3. Загрузить маршруты алгоритмов

Пример таблицы: `od_id=001, algorithm=shortest, geometry=LineString(...)` и `od_id=001, algorithm=proposed, geometry=LineString(...)`. Все алгоритмы должны решать одинаковые OD-пары.

In [ ]:
if not ROUTES_PATH.exists():
    raise FileNotFoundError(f"Положите маршруты в {ROUTES_PATH.resolve()}")
routes = gpd.read_file(ROUTES_PATH, layer=ROUTES_LAYER)
required = {"od_id", "algorithm", "geometry"}
if missing := required - set(routes.columns):
    raise ValueError(f"Нет полей: {sorted(missing)}")
if routes.crs is None:
    raise ValueError("У маршрутов не задан CRS")
if not routes.geometry.geom_type.isin(["LineString", "MultiLineString"]).all():
    raise ValueError("Геометрия маршрута должна быть LineString/MultiLineString")
routes = routes.to_crs(observed.crs).copy()
routes["route_length_m"] = routes.length
routes[["od_id", "algorithm", "route_length_m"]].head()

## 4. Сопоставить маршрут с наблюдёнными рёбрами

Маршрут дискретизируется каждые 10 м; каждой точке назначается ближайший STADTRADELN-сегмент не дальше 20 м. Это не полноценный map matching. Если алгоритм работает на том же OSM-графе, лучше экспортировать `osm_way_id` и сопоставлять рёбра точно.

In [ ]:
from shapely.ops import linemerge

def sample_routes(routes_gdf, step_m):
    records = []
    for row in routes_gdf.itertuples():
        line = linemerge(row.geometry) if row.geometry.geom_type == "MultiLineString" else row.geometry
        if line.geom_type != "LineString":
            continue
        distances = np.unique(np.arange(0, line.length + step_m, step_m).clip(max=line.length))
        for sequence, distance in enumerate(distances):
            records.append({"od_id": row.od_id, "algorithm": row.algorithm,
                            "sequence": sequence, "geometry": line.interpolate(distance)})
    return gpd.GeoDataFrame(records, geometry="geometry", crs=routes_gdf.crs)

route_points = sample_routes(routes, SAMPLE_STEP_M)
matched = gpd.sjoin_nearest(
    route_points, observed[["osm_way_id", "number_of_matched_trips", "geometry"]],
    how="left", max_distance=MAX_MATCH_DISTANCE_M, distance_col="match_distance_m")
matched = (matched.sort_values("match_distance_m")
           .drop_duplicates(["od_id", "algorithm", "sequence"]))
matched["is_matched"] = matched["number_of_matched_trips"].notna()
matched["flow"] = matched["number_of_matched_trips"].fillna(0)
matched.head()

## 5. Route-level метрики

Основная метрика — среднее `log(1 + flow)` вдоль маршрута: логарифм ослабляет влияние нескольких сверхпопулярных сегментов. Также считаются исходный flow, доля пути на верхней четверти рёбер и `match_rate`. Низкий `match_rate` означает, что маршрут плохо покрывается опубликованной сетью.

In [ ]:
popular_threshold = observed["number_of_matched_trips"].quantile(.75)
matched["log_flow"] = np.log1p(matched["flow"])
matched["on_popular_edge"] = matched["flow"] >= popular_threshold
metrics = (matched.groupby(["od_id", "algorithm"], as_index=False)
    .agg(popularity_log_mean=("log_flow", "mean"), flow_mean=("flow", "mean"),
         flow_median=("flow", "median"), popular_edge_share=("on_popular_edge", "mean"),
         match_rate=("is_matched", "mean"), mean_match_distance_m=("match_distance_m", "mean"))
    .merge(routes[["od_id", "algorithm", "route_length_m"]],
           on=["od_id", "algorithm"], how="left"))
metrics.head(20)

## 6. Сравнение с shortest baseline

Для каждого OD считаются удлинение и выигрыш популярности относительно кратчайшего маршрута. Получается Парето-компромисс: слева и выше — лучше.

In [ ]:
baseline = (metrics.loc[metrics["algorithm"].eq(BASELINE_ALGORITHM),
                        ["od_id", "route_length_m", "popularity_log_mean"]]
            .rename(columns={"route_length_m": "baseline_length_m",
                             "popularity_log_mean": "baseline_popularity"}))
if baseline["od_id"].duplicated().any():
    raise ValueError("Для OD должна быть одна baseline-строка")
comparison = metrics.merge(baseline, on="od_id", how="left", validate="many_to_one")
if comparison["baseline_length_m"].isna().any():
    raise ValueError("Не для всех OD найден shortest baseline")
comparison["detour_pct"] = 100 * (comparison["route_length_m"] / comparison["baseline_length_m"] - 1)
comparison["popularity_gain"] = comparison["popularity_log_mean"] - comparison["baseline_popularity"]
comparison.head()

In [ ]:
import plotly.express as px
fig = px.scatter(comparison, x="detour_pct", y="popularity_gain", color="algorithm",
                 hover_data=["od_id", "match_rate", "popular_edge_share"],
                 title="Удлинение маршрута и выигрыш популярности",
                 labels={"detour_pct": "Удлинение к shortest, %",
                         "popularity_gain": "Δ среднего log(1 + flow)"})
fig.add_hline(y=0, line_dash="dash"); fig.show()
summary = comparison.groupby("algorithm", as_index=False).agg(
    n_od=("od_id", "nunique"), median_detour_pct=("detour_pct", "median"),
    median_popularity_gain=("popularity_gain", "median"),
    mean_popularity=("popularity_log_mean", "mean"),
    mean_popular_edge_share=("popular_edge_share", "mean"),
    mean_match_rate=("match_rate", "mean"))
summary

## 7. Разделение по дням недели

Если popularity используется внутри алгоритма, оценка на тех же counts будет утечкой. Поля `number_of_matched_trips_monday` … `sunday` позволяют, например, строить веса по понедельнику–четвергу, а оценивать маршруты по пятнице–воскресенью.

In [ ]:
day_cols = [c for c in weekday if c.startswith("number_of_matched_trips_")]
weekend_cols = [c for c in day_cols if c.endswith(("friday", "saturday", "sunday"))]
train_cols = [c for c in day_cols if c not in weekend_cols]
weekday["train_flow_mon_thu"] = weekday[train_cols].sum(axis=1)
weekday["test_flow_fri_sun"] = weekday[weekend_cols].sum(axis=1)
weekday[["osm_way_id", "train_flow_mon_thu", "test_flow_fri_sun", "geometry"]].explore(
    column="test_flow_fri_sun", scheme="FisherJenks", k=7, cmap="magma",
    tiles="CartoDB positron", style_kwds={"weight": 3})

## 8. Экспорт и протокол

Рекомендуется сообщать: одинаковые OD для всех методов; shortest baseline; распределение по OD, а не только среднее; `match_rate`; popularity вместе с detour; train/test split, если поток входил в функцию стоимости. Высокая популярность может означать удобство, но также отсутствие альтернатив и смещение выборки STADTRADELN.

In [ ]:
OUTPUT_DIR = Path("outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
comparison.to_csv(OUTPUT_DIR / "stadtradeln_route_evaluation.csv", index=False)
summary.to_csv(OUTPUT_DIR / "stadtradeln_algorithm_summary.csv", index=False)
print("Сохранено в outputs/")